# NerGuard — Demo

**NerGuard** is an entropy-gated hybrid NER pipeline for privacy-compliant PII detection. It combines a multilingual mDeBERTa-v3 base model with optional LLM routing (OpenAI or Ollama) for uncertain spans.

- 📦 [PyPI](https://pypi.org/project/nerguard/) · 🤗 [Model on HuggingFace](https://huggingface.co/exdsgift/NerGuard-0.3B) · 💻 [GitHub](https://github.com/exdsgift/NerGuard)

**Runtime:** GPU recommended (T4 is fine). Go to **Runtime → Change runtime type → T4 GPU**.

## 1. Install

In [ ]:
!pip install -q nerguard

## 2. Basic usage — NER model only (no API key needed)

The NER model (~300 MB) downloads automatically from HuggingFace on first use.

In [ ]:
from nerguard import Redactor

ng = Redactor()

text = "Hi, I'm John Smith. My email is john.smith@acme.com and my SSN is 078-05-1120."
result = ng.redact(text)

print("Redacted:", result.text)
print("Mapping: ", result.mapping)
print("Entities:")
for e in result.entities:
    print(f"  [{e['label']}] '{e['text']}' (conf={e['confidence']:.3f}, source={e['source']})")

## 3. Batch redaction

In [ ]:
texts = [
    "Dear Maria García, your appointment is on 2024-03-15 at 10:30 AM.",
    "Please contact support at help@example.org or call +1-800-555-0199.",
    "Invoice for account IBAN DE89370400440532013000, tax ID 12-3456789.",
]

results = ng.redact_batch(texts)
for i, r in enumerate(results):
    print(f"[{i+1}] {r.text}")

## 4. LLM routing — higher accuracy on ambiguous spans (OpenAI)

LLM routing improves recall on uncertain predictions. To use it:

1. Open the 🔑 **Secrets** panel in the left sidebar (or go to **Tools → Secrets**).
2. Add a secret named `OPENAI_API_KEY` with your OpenAI key.
3. Enable **Notebook access** for that secret.

Then run the cell below.

In [ ]:
from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")

ng_llm = Redactor(
    llm_routing=True,
    llm_source="openai",
    llm_model="gpt-4o",
    api_key=openai_api_key,
)

# Ambiguous text where LLM routing helps
text = "Call me at 078-05-1120. My card ending in 4111 1111 1111 1111 is active."
result = ng_llm.redact(text)

print("Redacted:", result.text)
print("Mapping: ", result.mapping)

## 5. Generic placeholders (maximum compression)

Use `typed=False` to replace all PII with `[PII]` instead of typed markers — useful when you want maximum token compression for downstream LLM context windows.

In [ ]:
ng_generic = Redactor(typed=False)

text = "John Smith, born 1990-07-21, lives at 12 Baker Street, London."
result = ng_generic.redact(text)

print("Typed=False:", result.text)

## 6. Interactive redaction

Enter any text below and redact it on the fly.

In [ ]:
custom_text = "My name is Alice Dupont, I live in Paris and my phone is +33 6 12 34 56 78."  # @param {type:"string"}

result = ng.redact(custom_text)
print("Redacted:", result.text)
if result.entities:
    print("\nDetected entities:")
    for e in result.entities:
        print(f"  [{e['label']}] '{e['text']}' — conf={e['confidence']:.3f}")

## 7. LangChain Integration

NerGuard integrates with LangChain as both a **DocumentTransformer** (for RAG pipelines) and a **Tool** (for agent workflows).

Install the optional dependency:

```bash
pip install nerguard[langchain]
```

In [ ]:
!pip install -q langchain-core

### 7a. DocumentTransformer — anonymize documents in a RAG pipeline

`NerGuardAnonymizer` redacts PII from `Document.page_content` and stores the entity mapping in metadata for potential de-anonymization.

In [ ]:
from langchain_core.documents import Document
from nerguard.langchain import NerGuardAnonymizer

# Create sample documents
docs = [
    Document(page_content="John Smith's email is john.smith@acme.com"),
    Document(page_content="Maria García lives at 42 Baker Street, London"),
]

# Anonymize
anonymizer = NerGuardAnonymizer()
anon_docs = anonymizer.transform_documents(docs)

for i, doc in enumerate(anon_docs):
    print(f"[Doc {i+1}] {doc.page_content}")
    print(f"  Mapping: {doc.metadata['nerguard_mapping']}\n")

### 7b. Tool — let agents redact text on demand

`NerGuardTool` wraps `Redactor.redact()` as a LangChain `BaseTool` that agents can invoke.

In [ ]:
from nerguard.langchain import NerGuardTool

tool = NerGuardTool()

# Invoke the tool as an agent would
redacted = tool.invoke({"text": "Call Alice Dupont at +33 6 12 34 56 78"})
print("Redacted:", redacted)
print("Mapping:", tool.last_mapping)